<a href="https://colab.research.google.com/github/Vbenitez1/Inteligencia-Artificial/blob/main/MultiAgente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importacion de librerias

In [1]:
import pandas as pd
import numpy as np
import io
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report

print("Listo, librerias cargadas")

Listo, librerias cargadas


Generacion del DATASET

In [2]:
# Creamos los datos de ventas de productos tipicos de Paraguay
# Los datos tienen errores a proposito para simular datos reales

np.random.seed(42)
n = 200

productos     = ['Chipa', 'Sopa paraguaya', 'Mbeju', 'Chipa guasu', 'Pastel mandio']
departamentos = ['Central', 'Asuncion', 'Cordillera', 'Paraguari', 'Itapua']
vendedores    = ['Don Beto', 'Dona Rosa', 'Carlitos', 'La Pety', 'El Chuky']

data = {
    'producto'      : np.random.choice(productos, n),
    'departamento'  : np.random.choice(departamentos, n),
    'vendedor'      : np.random.choice(vendedores, n),
    'precio_gs'     : np.random.randint(1000, 50000, n).astype(float),  # precios en guaranies
    'cantidad'      : np.random.randint(1, 100, n).astype(float),
    'temperatura_c' : np.random.uniform(18, 42, n),                     # temperatura del dia
    'venta_alta'    : None
}

df_raw = pd.DataFrame(data)

# Si el ingreso total supera la mitad del promedio, la venta se considera "alta"
ingreso = df_raw['precio_gs'] * df_raw['cantidad']
df_raw['venta_alta'] = (ingreso > ingreso.median()).astype(int)

# --- Metemos errores como pasa en datos reales ---

# Algunos precios y cantidades sin valor
idx_nan_precio = np.random.choice(n, 15, replace=False)
idx_nan_cant   = np.random.choice(n, 10, replace=False)
df_raw.loc[idx_nan_precio, 'precio_gs'] = np.nan
df_raw.loc[idx_nan_cant,   'cantidad']  = np.nan

# Algunos departamentos en mayusculas (inconsistencia)
idx_case = np.random.choice(n, 20, replace=False)
df_raw.loc[idx_case, 'departamento'] = df_raw.loc[idx_case, 'departamento'].str.upper()

# Temperaturas imposibles
df_raw.loc[[5, 77], 'temperatura_c'] = [999, -50]

# Guardamos como CSV en memoria
csv_buffer = io.StringIO()
df_raw.to_csv(csv_buffer, index=False)
csv_buffer.seek(0)

print(f"Dataset generado: {df_raw.shape[0]} filas y {df_raw.shape[1]} columnas")
print(f"Valores faltantes:\n{df_raw.isnull().sum()}")
df_raw.head(8)

Dataset generado: 200 filas y 7 columnas
Valores faltantes:
producto          0
departamento      0
vendedor          0
precio_gs        15
cantidad         10
temperatura_c     0
venta_alta        0
dtype: int64


,producto,departamento,vendedor,precio_gs,cantidad,temperatura_c,venta_alta
0,Chipa guasu,Asuncion,Carlitos,12003.0,47.0,32.978437,0
1,Pastel mandio,Cordillera,El Chuky,22732.0,86.0,30.954745,1
2,Mbeju,Central,Don Beto,26826.0,56.0,28.529872,1
3,Pastel mandio,Central,El Chuky,31354.0,94.0,31.859671,1
4,Pastel mandio,PARAGUARI,La Pety,14843.0,63.0,26.528699,1
5,Sopa paraguaya,Cordillera,El Chuky,NaN,48.0,999.000000,1
6,Mbeju,Itapua,Don Beto,49529.0,61.0,30.764580,1
7,Mbeju,Cordillera,La Pety,7190.0,NaN,19.598859,0
